# Case 7 v5 — классификация товарных групп по RD-документам

Нужно оставить среди шумных кандидатов документы трёх групп:

| TG | Что ищем |
|---:|---|
| 4 | парфюмерия |
| 35 | косметика, бытовая химия, личная гигиена |
| 43 | моторные масла |

Бытовой пример: `tg_ids` — это широкий поисковый запрос. Он почти не пропускает нужное, но приносит много лишнего. Наша модель — второй фильтр, который убирает ложные срабатывания и обязан сохранить **Recall ≥ 97%**.

Главные правила честности:

1. `Truth` задаёт только ответы, но его служебные поля не становятся признаками.
2. Все товары (`rank`) одного документа объединяются.
3. Порог выбирается на validation и больше не меняется.
4. После этого test открывается один раз.

## 1. Импорты и настройки

Используем только классический стек. `RANDOM_STATE` фиксирует случайность, а версии библиотек попадут в отчёт.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import gc, hashlib, json, math, os, re, sys, time, unicodedata, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from scipy import sparse
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
TARGETS = (4, 35, 43)
CACHE_VERSION = 5  # v5: rebuilt on corrected Truth (2_truth...), including TG4 linkage
TARGET_NAMES = {4: 'Парфюмерия', 35: 'Косметика и бытовая химия', 43: 'Моторные масла'}
V1 = {
    4:  {'precision': .1532628218, 'recall': .9833546735, 'f1': .2651933702},
    35: {'precision': .5094487163, 'recall': .97725,      'f1': .6697507068},
    43: {'precision': .9064891847, 'recall': .9711229947, 'f1': .9376936317},
}

ROOT = Path.cwd().parent
PROJECT = ROOT

if PROJECT.name != 'v3_pandas':
    PROJECT = PROJECT / 'v3_pandas'

MAIN_PATH = ROOT / "data" / "case_7_data_for_rd.snappy.parquet"
TRUTH_PATH = ROOT / "data" / "2_truth_rd_data.snappy.parquet"
DEFINITIONS_PATH = ROOT / "data" / "TG_Definitions.xlsx"

# v4 uses a separate artifact namespace so locked/legacy v4 files cannot be reused.
ARTIFACTS = PROJECT / 'artifacts_v5'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

PREPARED_PATH = ARTIFACTS / 'prepared_documents.parquet'
BUNDLE_PATH = ARTIFACTS / 'solution_bundle.joblib'
REPORT_PATH = ARTIFACTS / 'report.json'

for path in (MAIN_PATH, TRUTH_PATH, DEFINITIONS_PATH):
    assert path.exists(), f'File not found: {path}'
INPUT_FINGERPRINT = hashlib.sha256(
    '|'.join(f'{path.name}:{path.stat().st_size}:{path.stat().st_mtime_ns}'
             for path in (MAIN_PATH, TRUTH_PATH, DEFINITIONS_PATH)).encode()
).hexdigest()

versions = {'python': sys.version.split()[0], 'pandas': pd.__version__,
            'numpy': np.__version__, 'pyarrow': pa.__version__}
print(versions)
print('Основной parquet:', round(MAIN_PATH.stat().st_size / 2**30, 2), 'GiB')

{'python': '3.13.11', 'pandas': '3.0.5', 'numpy': '2.5.1', 'pyarrow': '25.0.0'}
Основной parquet: 2.22 GiB


## 0.1. Источник разметки v5

Критически важно: этот ноутбук **не использует старый** `truth_rd_data.snappy.parquet`.
Финальная пересборка должна идти только с `data/2_truth_rd_data.snappy.parquet` — новым Truth, где TG4 имеет исправленный `rd_number` и добавлен `rd_date`.

Перед расчётом `INPUT_FINGERPRINT` наличие этого файла проверяется через `assert path.exists()`.


## 2. Почему здесь два теста

**Legacy benchmark** повторяет выборку и random split v1. Только здесь сравнение с v1 методологически сопоставимо.

**Full candidate-test** ближе к production: для каждой TG берём всех её кандидатов плюс все положительные документы Truth. Это значительно сложнее и честнее. Старые проценты v1 рядом с ним показываются только как справка — выборки разные.

In [2]:
print('Legacy: те же sampling/caps/random_state, что у v1.')
print('Strict: все кандидаты конкретной TG + все её positives; grouped split.')

Legacy: те же sampling/caps/random_state, что у v1.
Strict: все кандидаты конкретной TG + все её positives; grouped split.


## 3. Нормализация номера и Truth

Используем **новый исправленный Truth** `2_truth_rd_data.snappy.parquet`.
Из Truth берём только `tg` и номер документа для разметки; `rd_date` используется только для аудита linkage и не становится признаком модели.
`code_tnved`, `rd_type` и `rd_date` не подаются в модель, чтобы не дать ей прямую подсказку из Truth.

In [3]:
def normalize_rd_number(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ''
    value = unicodedata.normalize('NFKC', str(value)).upper()
    return ''.join(ch for ch in value if ch.isalnum())

def stable_int(value, modulo=10_000):
    digest = hashlib.md5(str(value).encode('utf-8', errors='ignore')).hexdigest()
    return int(digest[:12], 16) % modulo

truth_raw = pd.read_parquet(TRUTH_PATH, columns=['tg', 'rd_number', 'rd_date'])
truth_raw['rd_norm'] = truth_raw['rd_number'].map(normalize_rd_number)
truth_raw['tg'] = pd.to_numeric(truth_raw['tg'], errors='coerce').astype('Int64')
truth_raw['rd_date'] = pd.to_datetime(truth_raw['rd_date'], errors='coerce')
truth_raw = truth_raw[truth_raw.tg.isin(TARGETS) & truth_raw.rd_norm.ne('')]
truth_by_rd = (truth_raw.groupby('rd_norm').tg
               .agg(lambda s: tuple(sorted(set(map(int, s)))))
               .rename('truth_tgs'))
truth_lookup = truth_by_rd.to_dict()
truth_set = set(truth_lookup)

assert normalize_rd_number(' RU-С-RU.АЯ01.В.001 ') == normalize_rd_number('ru с ru ая01 в001')
assert truth_raw.rd_norm.map(normalize_rd_number).eq(truth_raw.rd_norm).all()
multi_truth = int(truth_by_rd.map(len).gt(1).sum())
print(f'Truth: {len(truth_raw):,} строк, {len(truth_by_rd):,} документов, multi-label: {multi_truth}')
display(truth_raw.tg.value_counts().sort_index().rename('rows').to_frame())

Truth: 577,229 строк, 97,818 документов, multi-label: 77


,rows
tg,
4,4455
35,349134
43,223640


## 4. Потоковое чтение основного parquet (~2.22 GiB)

`pandas.read_parquet` целиком здесь опасен для RAM. PyArrow читает по одному из 18 row groups.

Чтение двухпроходное:

1. сначала находим номера всех Truth, всех target-кандидатов и 0,1% стабильного background;
2. затем забираем **все rank выбранных номеров** и извлекаем компактные поля JSON.

Второй пункт важен: если выбирать строки по отдельности, у документа снова потеряется товар.

In [4]:
def clean_text(value, limit=8000):
    if value is None:
        return ''
    if isinstance(value, dict):
        value = ' '.join(clean_text(v, limit) for v in value.values())
    elif isinstance(value, (list, tuple)):
        value = ' '.join(clean_text(v, limit) for v in value)
    value = unicodedata.normalize('NFKC', str(value)).lower()
    value = re.sub(r'([\W_])\1{3,}', r'\1\1', value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value[:limit]

def extract_tnved(value):
    text = clean_text(value, 4000)
    return tuple(sorted(set(re.findall(r'(?<!\d)\d{4,10}(?!\d)', text))))

def json_record(raw):
    try:
        p = json.loads(raw) if raw else {}
        valid = isinstance(p, dict)
        if not valid: p = {}
    except (TypeError, json.JSONDecodeError):
        p, valid = {}, False
    product = p.get('product') if isinstance(p.get('product'), dict) else {}
    decl = p.get('declaration') if isinstance(p.get('declaration'), dict) else {}
    cert = p.get('certificate') if isinstance(p.get('certificate'), dict) else {}
    applicant = p.get('applicant') if isinstance(p.get('applicant'), dict) else {}
    maker = p.get('manufacturer') if isinstance(p.get('manufacturer'), dict) else {}
    body = p.get('certificationBody') if isinstance(p.get('certificationBody'), dict) else {}
    product_text = clean_text([p.get('nameProd'), product.get('productName'),
                               product.get('productInfo'), product.get('identification')])
    context_text = clean_text([p.get('useArea'), p.get('docNorm'), p.get('protocol'),
                               p.get('techRegulations'), cert.get('issueBasis')])
    party_text = clean_text([p.get('firmMadeName'), p.get('firmGetName'),
                             applicant.get('fullName'), applicant.get('address'),
                             maker.get('name'), maker.get('address'), body.get('name')], 5000)
    return {
        'json_valid': int(valid), 'product_text': product_text,
        'context_text': context_text, 'party_text': party_text,
        'tnved_codes': ','.join(extract_tnved(product.get('tnved'))),
        'doc_type': str(p.get('type') or ''), 'status': str(p.get('statusGroup') or ''),
        'active': str(p.get('active') if p.get('active') is not None else ''),
        'active_from': str(p.get('activeFromDate') or p.get('dateFrom') or ''),
        'decl_scheme': str(decl.get('idDeclScheme') or ''),
        'decl_type': str(decl.get('idDeclType') or ''),
        'decl_from': str(decl.get('declRegDate') or ''),
        'decl_to': str(decl.get('declEndDate') or ''),
        'cert_scheme': str(cert.get('idCertScheme') or ''),
        'cert_type': str(cert.get('idCertType') or ''),
        'cert_from': str(cert.get('certRegDate') or ''),
        'cert_to': str(cert.get('certEndDate') or ''),
        'has_declaration': int(bool(decl)), 'has_certificate': int(bool(cert)),
        'has_applicant': int(bool(applicant)), 'has_manufacturer': int(bool(maker)),
    }

def scan_and_prepare():
    pf = pq.ParquetFile(MAIN_PATH)
    selected_docs = set()
    candidate_rows = Counter()
    print('Проход 1/2: выбираем номера документов')
    for rg in range(pf.num_row_groups):
        frame = pf.read_row_group(rg, columns=['rd_documentnumber', 'tg_ids']).to_pandas()
        norms = frame.rd_documentnumber.map(normalize_rd_number)
        cands = frame.tg_ids.map(lambda x: tuple(map(int, x)) if x is not None else ())
        target = cands.map(lambda x: bool(set(x) & set(TARGETS)))
        background = (~target) & norms.map(lambda x: stable_int(x) < 10)
        keep = norms.isin(truth_set) | target | background
        selected_docs.update(norms[keep & norms.ne('')])
        for tg in TARGETS:
            candidate_rows[tg] += int(cands.map(lambda x, g=tg: g in x).sum())
        print(f'  group {rg + 1:02d}/{pf.num_row_groups}: selected IDs {len(selected_docs):,}')

    temp_path = ARTIFACTS / '_selected_rows.tmp.parquet'
    writer = None
    print('Проход 2/2: читаем все rank выбранных документов')
    for rg in range(pf.num_row_groups):
        frame = pf.read_row_group(rg).to_pandas()
        frame['rd_norm'] = frame.rd_documentnumber.map(normalize_rd_number)
        frame = frame[frame.rd_norm.isin(selected_docs)].copy()
        rows = []
        for row in frame.itertuples(index=False):
            rec = json_record(row.rd_data)
            rec.update({
                'rd_norm': row.rd_norm, 'rd_number': str(row.rd_documentnumber or ''),
                'rank': int(row.rank) if pd.notna(row.rank) else -1,
                'candidate_tgs': ','.join(map(str, sorted(set(list(row.tg_ids) if row.tg_ids is not None else [])))),
            })
            rows.append(rec)
        compact = pd.DataFrame.from_records(rows)
        table = pa.Table.from_pandas(compact, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(temp_path, table.schema, compression='snappy')
        writer.write_table(table)
        print(f'  group {rg + 1:02d}/{pf.num_row_groups}: rows {len(compact):,}')
    if writer is not None: writer.close()

    raw = pd.read_parquet(temp_path)
    counts = raw.rd_norm.value_counts()
    single_ids = set(counts[counts.eq(1)].index)
    singles = raw[raw.rd_norm.isin(single_ids)].copy()

    def unique_join(series, sep=' | ', limit=30000):
        seen, out = set(), []
        for item in series:
            item = str(item or '').strip()
            if item and item not in seen:
                seen.add(item); out.append(item)
        return sep.join(out)[:limit]

    def union_csv(series):
        values = set()
        for item in series:
            values.update(x for x in str(item or '').split(',') if x)
        return ','.join(sorted(values, key=lambda x: (len(x), x)))

    duplicate = raw[~raw.rd_norm.isin(single_ids)]
    aggregated = []
    for rd_norm, g in duplicate.groupby('rd_norm', sort=False):
        first = g.iloc[0].to_dict()
        for col in ['product_text', 'context_text', 'party_text']:
            first[col] = unique_join(g[col])
        first['tnved_codes'] = union_csv(g.tnved_codes)
        first['candidate_tgs'] = union_csv(g.candidate_tgs)
        first['rank'] = ','.join(map(str, sorted(set(g['rank'].astype(int)))))
        first['row_count'] = len(g)
        first['rank_count'] = g['rank'].nunique()
        first['product_count'] = g.product_text.replace('', np.nan).nunique(dropna=True)
        first['conflicting_products'] = int(first['product_count'] > 1)
        first['json_valid'] = int(g.json_valid.all())
        for col in ['has_declaration', 'has_certificate', 'has_applicant', 'has_manufacturer']:
            first[col] = int(g[col].max())
        aggregated.append(first)
    multi_docs = pd.DataFrame.from_records(aggregated)

    singles['rank'] = singles['rank'].astype(str)
    singles['row_count'] = 1; singles['rank_count'] = 1
    singles['product_count'] = singles.product_text.ne('').astype(int)
    singles['conflicting_products'] = 0
    docs = pd.concat([singles, multi_docs], ignore_index=True, sort=False)
    docs['truth_tgs'] = docs.rd_norm.map(lambda x: ','.join(map(str, truth_lookup.get(x, ()))))
    docs['cache_version'] = CACHE_VERSION
    docs['input_fingerprint'] = INPUT_FINGERPRINT
    temp_path.unlink()
    return docs, dict(candidate_rows)

if PREPARED_PATH.exists():
    cache_names = set(pq.ParquetFile(PREPARED_PATH).schema_arrow.names)
    cached_feature_names = {
        'text_len','product_len','context_len','party_len','digit_ratio','latin_ratio','cyrillic_ratio',
        'candidate_count','tnved_count','rd_len','rd_segments','rd_prefix','rd_country','issue_year',
        'missing_date','field_count','term_conflict','product_signature','split_bucket','strict_split','strict_in_scope'
    } | {f'{kind}_{tg}' for tg in TARGETS for kind in ('candidate','tnved_prefix','tnved_exact','term_score')}
    cache_columns = sorted((cached_feature_names | {
        'cache_version','input_fingerprint','json_valid','product_text','context_text','doc_type','status','active',
        'decl_scheme','decl_type','cert_scheme','cert_type','has_declaration','has_certificate',
        'has_applicant','has_manufacturer','party_text','tnved_codes','active_from','rd_norm','rd_number','rank','candidate_tgs','truth_tgs',
        'row_count','rank_count','product_count','conflicting_products'
    }) & cache_names)
    documents = pd.read_parquet(PREPARED_PATH, columns=cache_columns, dtype_backend='pyarrow')
    FEATURES_CACHED = {'term_score_4','term_score_35','term_score_43'}.issubset(cache_names)
    SPLIT_CACHED = {'product_signature','strict_split','strict_in_scope'}.issubset(cache_names)
    cache_valid = ('cache_version' in documents and 'input_fingerprint' in documents and
                   documents.cache_version.eq(CACHE_VERSION).all() and
                   documents.input_fingerprint.eq(INPUT_FINGERPRINT).all())
    if cache_valid:
        candidate_row_counts = {}
        print('Loaded valid document-level cache:', f'{len(documents):,}')
    else:
        # The exact named cache is stale; regenerate it from the three declared inputs.
        del documents
        PREPARED_PATH.unlink()
        started = time.time()
        documents, candidate_row_counts = scan_and_prepare()
        documents.to_parquet(PREPARED_PATH, index=False, compression='snappy')
        print('Regenerated stale document-level cache in', round((time.time() - started) / 60, 1), 'min.')
        FEATURES_CACHED = False; SPLIT_CACHED = False
else:
    started = time.time()
    documents, candidate_row_counts = scan_and_prepare()
    documents.to_parquet(PREPARED_PATH, index=False, compression='snappy')
    print('Первичная витрина сохранена за', round((time.time() - started) / 60, 1), 'мин.')
    FEATURES_CACHED = False; SPLIT_CACHED = False

def csv_set(value):
    return frozenset(int(x) for x in str(value or '').split(',') if x)

documents['candidate_set'] = documents.candidate_tgs.map(csv_set)
documents['truth_set'] = documents.truth_tgs.map(csv_set)
documents['tnved_set'] = (documents.tnved_codes.map(
    lambda x: frozenset(v for v in str(x or '').split(',') if v))
    if 'tnved_codes' in documents else pd.Series([frozenset()] * len(documents), index=documents.index))
print('Документов в витрине:', f'{len(documents):,}')

Проход 1/2: выбираем номера документов
  group 01/18: selected IDs 288,794
  group 02/18: selected IDs 359,810
  group 03/18: selected IDs 449,572
  group 04/18: selected IDs 487,566
  group 05/18: selected IDs 551,669
  group 06/18: selected IDs 646,503
  group 07/18: selected IDs 739,827
  group 08/18: selected IDs 798,180
  group 09/18: selected IDs 837,945
  group 10/18: selected IDs 874,490
  group 11/18: selected IDs 917,782
  group 12/18: selected IDs 963,454
  group 13/18: selected IDs 1,011,063
  group 14/18: selected IDs 1,055,005
  group 15/18: selected IDs 1,079,206
  group 16/18: selected IDs 1,196,278
  group 17/18: selected IDs 1,365,568
  group 18/18: selected IDs 1,417,029
Проход 2/2: читаем все rank выбранных документов
  group 01/18: rows 288,795
  group 02/18: rows 71,038
  group 03/18: rows 89,773
  group 04/18: rows 38,195
  group 05/18: rows 64,143
  group 06/18: rows 94,973
  group 07/18: rows 93,357
  group 08/18: rows 58,353
  group 09/18: rows 42,572
  group 

## 5. Почему нельзя выбирать минимальный `rank`

`rank` — это строка товара внутри РД. У одного номера могут быть разные продукты. v1 оставлял только минимальный rank, поэтому часть содержимого исчезала.

Ниже проверяем, что после агрегации сохранились все rank, кандидаты и товары.

In [5]:
assert documents.rd_norm.is_unique
assert documents.loc[documents.row_count.gt(1), 'rank_count'].ge(1).all()
assert documents.loc[documents.conflicting_products.eq(1), 'product_count'].gt(1).all()
covered_multi = sum(len(truth_lookup.get(rd, ())) > 1 for rd in documents.rd_norm)
assert documents.truth_set.map(len).gt(1).sum() == covered_multi

coverage = len(set(documents.rd_norm) & truth_set) / max(len(truth_set), 1)
audit = {
    'documents': int(len(documents)), 'truth_documents': int(len(truth_set)),
    'truth_coverage': float(coverage), 'covered_truth_documents': int(len(set(documents.rd_norm) & truth_set)),
    'multi_label_truth': multi_truth, 'covered_multi_label_truth': int(covered_multi),
    'invalid_json_documents': int(documents.json_valid.eq(0).sum()),
    'multi_row_documents': int(documents.row_count.gt(1).sum()),
    'conflicting_product_documents': int(documents.conflicting_products.sum()),
    'candidate_rows': candidate_row_counts,
}
display(pd.Series(audit).rename('value').to_frame())
display(documents.loc[documents.conflicting_products.eq(1),
                      ['rd_number', 'rank', 'product_count', 'product_text']].head(3))

,value
documents,1417029
truth_documents,97818
truth_coverage,0.882261
covered_truth_documents,86301
multi_label_truth,77
covered_multi_label_truth,66
invalid_json_documents,0
multi_row_documents,3732
conflicting_product_documents,3718
candidate_rows,"{4: 223365, 35: 1330737, 43: 102070}"


,rd_number,rank,product_count,product_text
1413297,ВП RU Д-CN.РА01.А.30884/23,"1,2",2,компоненты транспортных средств: шины пневмати...
1413298,ВП RU Д-ES.РА01.А.52788/23,"1,2",2,elizabeth arden green tea лосьон для тела 100м...
1413299,Д-RU.ЛФ03.А.00490,"1,2",2,лифт пассажирский электрический для зданий леч...


## 6. Словари из `TG_Definitions.xlsx`

Excel читается прямо здесь. Код ищет в строке ТН ВЭД, определяет группу по префиксу, а соседний текст превращает в термины. Ручные короткие сигналы дополняют Excel, но не заменяют его.

In [6]:
# Candidate/feature rules only. They never write truth_tgs or truth_lookup.
GROUP_PREFIXES = {4: ('3303',), 35: ('3304', '3305', '3306', '3307', '3808', '3401', '3402'),
                  43: ('2710', '3403')}
MANUAL_TERMS = {
    4: ['духи', 'парфюмерная вода', 'туалетная вода', 'одеколон', 'eau de parfum', 'eau de toilette'],
    35: ['крем', 'шампунь', 'бальзам', 'лосьон', 'сыворотка', 'дезодорант', 'антиперспирант',
         'зубная паста', 'гель для душа', 'жидкое мыло', 'моющее средство', 'косметическая продукция'],
    43: ['моторное масло', 'масло моторное', 'engine oil', 'lubricant', 'смазочный материал',
         'sae 5w', 'sae 10w', 'полусинтетическое масло', 'синтетическое масло'],
}

def group_for_code(code):
    digits = re.sub(r'\D', '', str(code))
    for tg, prefixes in GROUP_PREFIXES.items():
        if any(digits.startswith(p) for p in prefixes): return tg
    return None

definition_terms = {tg: set(MANUAL_TERMS[tg]) for tg in TARGETS}
definition_codes = {tg: set() for tg in TARGETS}
xls = pd.ExcelFile(DEFINITIONS_PATH)
for sheet in xls.sheet_names:
    table = pd.read_excel(xls, sheet_name=sheet, header=None)
    for row in table.itertuples(index=False, name=None):
        joined = ' | '.join(clean_text(x, 1000) for x in row if pd.notna(x))
        codes = re.findall(r'(?<!\d)\d{4,10}(?!\d)', joined.replace(' ', ''))
        groups = {group_for_code(c) for c in codes} - {None}
        for tg in groups:
            definition_codes[tg].update(c for c in codes if group_for_code(c) == tg)
            for cell in row:
                if pd.isna(cell): continue
                cleaned = re.sub(r'\d[\d .-]{2,}', ' ', clean_text(cell, 500))
                for part in re.split(r'[;,|()]', cleaned):
                    part = re.sub(r'\s+', ' ', part).strip(' .:-')
                    words = part.split()
                    if 1 <= len(words) <= 7 and 4 <= len(part) <= 80:
                        definition_terms[tg].add(part)

# Убираем слишком общие слова: они дают больше шума, чем пользы.
for tg in TARGETS:
    definition_terms[tg] = sorted(t for t in definition_terms[tg]
                                  if len(t) >= 4 and t not in {'продукция', 'изделия', 'средства'})[:500]
    definition_codes[tg] = sorted(definition_codes[tg])

display(pd.DataFrame({
    'TG': TARGETS,
    'Терминов': [len(definition_terms[tg]) for tg in TARGETS],
    'Кодов': [len(definition_codes[tg]) for tg in TARGETS],
    'Пример': [', '.join(definition_terms[tg][:5]) for tg in TARGETS],
}))

,TG,Терминов,Кодов,Пример
0,4,9,4,"eau de parfum, eau de toilette, духи, духи и т..."
1,35,500,25,"bb-крем, bb-мусс, bb-средство, bb-флюид, cc-крем"
2,43,78,5,"engine oil, lubricant, sae 10w, sae 5w, вид то..."


## 7. Понятные структурные и доменные признаки

TF-IDF хорошо читает текст, но не обязан сам догадаться, что `3303` — код парфюмерии. Поэтому явно считаем:

- совпадения ТН ВЭД целиком и по префиксу;
- словарные сигналы каждой TG и конфликты между ними;
- состав candidates;
- длины и алфавит текста;
- даты, тип документа и заполненность JSON;
- число строк, rank, товаров и кодов.

In [7]:
def embedded_id_linkage_audit(truth_frame, document_ids):
    """Diagnostic only: discover complete main IDs embedded in malformed truth text."""
    doc_by_tail = defaultdict(set)
    for doc_id in document_ids:
        if len(doc_id) >= 12:
            doc_by_tail[doc_id[-7:]].add(doc_id)
    links = []
    unique_truth = truth_frame.drop_duplicates('rd_number')
    for row in unique_truth.itertuples(index=False):
        if int(row.tg) != 4:
            continue
        text_norm = normalize_rd_number(row.rd_number)
        possible = set()
        for start in range(max(len(text_norm) - 6, 0)):
            possible.update(doc_by_tail.get(text_norm[start:start + 7], ()))
        for doc_id in possible:
            if doc_id in text_norm:
                links.append((doc_id, row.rd_number))
    return links

truth_linkage_rows = []
for tg in TARGETS:
    truth_ids = set(truth_by_rd[truth_by_rd.map(lambda labels, g=tg: g in labels)].index)
    matched = truth_ids & set(documents.rd_norm)
    lengths = truth_raw.loc[truth_raw.tg.eq(tg), 'rd_norm'].str.len()
    truth_linkage_rows.append({
        'tg': tg, 'truth_rows': int(truth_raw.tg.eq(tg).sum()),
        'truth_documents': len(truth_ids), 'matched_documents': len(matched),
        'coverage': len(matched) / max(len(truth_ids), 1),
        'rd_norm_median_length': float(lengths.median()),
        'rd_norm_max_length': int(lengths.max()),
        'supervised_status': 'linked' if matched else 'unlinked_source_field',
    })
truth_linkage_audit = pd.DataFrame(truth_linkage_rows).set_index('tg')
tg4_embedded_id_links = embedded_id_linkage_audit(truth_raw, documents.rd_norm)
# New Truth is the source of labels. Embedded-ID matching remains diagnostics only.
assert 3808 not in truth_lookup
display(truth_linkage_audit)
print('TG4 exact embedded-ID links (diagnostic only):', len(tg4_embedded_id_links))


,truth_rows,truth_documents,matched_documents,coverage,rd_norm_median_length,rd_norm_max_length,supervised_status
tg,,,,,,,
4,4455,3942,3936,0.998478,22.0,30,linked
35,349134,77413,68370,0.883185,22.0,6273,linked
43,223640,16540,14061,0.850121,22.0,283,linked


TG4 exact embedded-ID links (diagnostic only): 3946


In [8]:
def rd_prefix(value):
    pieces = re.split(r'[. /-]+', str(value).upper())
    return (pieces[0] if pieces else '')[:12]

def rd_country(value):
    match = re.search(r'(?:RU|ЕАЭС)[. -]([A-ZА-Я]{2})', str(value).upper())
    return match.group(1) if match else ''

def add_features(df):
    df = df.copy()
    df['all_text'] = (df.product_text.fillna('') + ' ' + df.context_text.fillna('') +
                      ' ' + df.party_text.fillna('')).str.strip()
    df['field_text'] = ('prod__ ' + df.product_text.fillna('') + ' context__ ' +
                        df.context_text.fillna('') + ' party__ ' + df.party_text.fillna(''))
    length = df.all_text.str.len().clip(lower=1)
    df['text_len'] = length
    df['product_len'] = df.product_text.str.len()
    df['context_len'] = df.context_text.str.len()
    df['party_len'] = df.party_text.str.len()
    df['digit_ratio'] = df.all_text.str.count(r'\d') / length
    df['latin_ratio'] = df.all_text.str.count(r'[a-z]') / length
    df['cyrillic_ratio'] = df.all_text.str.count(r'[а-яё]') / length
    df['candidate_count'] = df.candidate_set.map(len)
    df['tnved_count'] = df.tnved_set.map(len)
    df['rd_len'] = df.rd_number.str.len()
    df['rd_segments'] = df.rd_number.str.count(r'[./-]') + 1
    df['rd_prefix'] = df.rd_number.map(rd_prefix)
    df['rd_country'] = df.rd_number.map(rd_country)
    date = pd.to_datetime(df.active_from, errors='coerce')
    df['issue_year'] = date.dt.year.fillna(0).astype('int16')
    df['missing_date'] = date.isna().astype('int8')
    df['field_count'] = df[['product_text','context_text','party_text']].ne('').sum(axis=1)
    for tg in TARGETS:
        df[f'candidate_{tg}'] = df.candidate_set.map(lambda s, g=tg: int(g in s)).astype('int8')
        prefixes = GROUP_PREFIXES[tg]
        exact = set(definition_codes[tg])
        df[f'tnved_prefix_{tg}'] = df.tnved_set.map(
            lambda codes, p=prefixes: sum(any(c.startswith(x) for x in p) for c in codes)).astype('int16')
        df[f'tnved_exact_{tg}'] = df.tnved_set.map(
            lambda codes, e=exact: sum(c in e for c in codes)).astype('int16')
        # Берём короткие уникальные термины: длинные определения медленны и редко совпадают дословно.
        foreign = set().union(*(definition_terms[g] for g in TARGETS if g != tg))
        excel_unique = [x for x in definition_terms[tg]
                        if x not in foreign and len(x.split()) <= 4 and len(x) <= 45]
        useful = list(dict.fromkeys(MANUAL_TERMS[tg] + excel_unique))[:60]
        pattern = '|'.join(
    re.escape(x.lower())
    for x in useful
)
        df[f'term_score_{tg}'] = (
    df["product_text"]
    .fillna("")
    .astype("string")
    .str.lower()
    .str.count(pattern)
    .clip(0, 20)
    .astype("int16")
)
    scores = df[[f'term_score_{tg}' for tg in TARGETS]].to_numpy()
    df['term_conflict'] = (np.sum(scores > 0, axis=1) > 1).astype('int8')
    return df

if FEATURES_CACHED:
    print('Структурные и словарные признаки загружены из кеша.')
else:
    started = time.time()
    documents = add_features(documents)
    cache_cols = [c for c in documents.columns if c not in {'candidate_set','truth_set','tnved_set'}]
    documents[cache_cols].to_parquet(PREPARED_PATH, index=False, compression='snappy')
    print('Признаки рассчитаны и добавлены в кеш за', round((time.time()-started)/60, 1), 'мин.')
TRUTH_ONLY = {'code_tnved', 'group_tnved', 'rd_type', 'rd_date'}
assert not (TRUTH_ONLY & set(documents.columns))

STRUCT_COLS = [
    'row_count','rank_count','product_count','conflicting_products','json_valid',
    'has_declaration','has_certificate','has_applicant','has_manufacturer',
    'text_len','product_len','context_len','party_len','digit_ratio','latin_ratio','cyrillic_ratio',
    'candidate_count','tnved_count','rd_len','rd_segments','issue_year','missing_date','field_count','term_conflict',
] + [f'{kind}_{tg}' for tg in TARGETS for kind in ('candidate','tnved_prefix','tnved_exact','term_score')]
CAT_COLS = ['doc_type','status','active','decl_scheme','decl_type','cert_scheme','cert_type','rd_prefix','rd_country']

display(documents.groupby(documents.truth_set.map(lambda s: ','.join(map(str, sorted(s))) or 'нет'))[
    [f'term_score_{tg}' for tg in TARGETS] + [f'tnved_prefix_{tg}' for tg in TARGETS]
].mean().round(2).head(10))

Признаки рассчитаны и добавлены в кеш за 2.4 мин.


,term_score_4,term_score_35,term_score_43,tnved_prefix_4,tnved_prefix_35,tnved_prefix_43
truth_set,,,,,,
35,0.00,1.62,0.06,0.00,0.84,0.00
"35,43",0.00,0.64,0.42,0.00,0.36,0.15
4,4.20,0.11,0.00,1.05,0.02,0.00
"4,35",3.50,1.16,0.00,0.59,0.56,0.00
"4,43",0.00,0.00,0.00,0.00,0.00,1.00
43,0.00,0.04,2.10,0.00,0.00,1.17
нет,0.07,0.54,0.06,0.02,0.27,0.04


## 8. Split без утечки похожих карточек

Обычный random split может положить почти одинаковые карточки одного товара в train и test. Создаём продуктовую сигнатуру из нормализованного названия, производителя и ТН ВЭД. Хеш сигнатуры задаёт часть:

- 0–49: fit;
- 50–59: calibration (внутри 60% train);
- 60–79: validation;
- 80–99: test.

Номер документа уникален после агрегации, а одна сигнатура всегда имеет ровно один split.

In [9]:
def product_signature(row):
    product = re.sub(r'\b\d+[.,]?\d*\b', '#', clean_text(row.product_text, 1200))
    product = re.sub(r'\s+', ' ', product).strip()
    party = clean_text(row.party_text, 250)[:120]
    codes = ','.join(sorted(row.tnved_set))
    base = product + '|' + party + '|' + codes
    return hashlib.md5(base.encode('utf-8', errors='ignore')).hexdigest()

if SPLIT_CACHED:
    print('Продуктовый split загружен из кеша.')
else:
    documents['product_signature'] = [product_signature(r) for r in documents.itertuples(index=False)]
    documents['split_bucket'] = documents.product_signature.map(lambda x: stable_int(x, 100)).astype('int8')
    documents['strict_split'] = pd.cut(documents.split_bucket, [-1,49,59,79,99],
                                      labels=['fit','calibration','validation','test']).astype(str)
    documents['strict_in_scope'] = (documents.candidate_set.map(lambda s: bool(s & set(TARGETS))) |
                                    documents.truth_set.map(lambda s: bool(s & set(TARGETS))))
    cache_cols = [c for c in documents.columns if c not in {'candidate_set','truth_set','tnved_set'}]
    documents[cache_cols].to_parquet(PREPARED_PATH, index=False, compression='snappy')

assert documents.groupby('product_signature').strict_split.nunique().max() == 1
assert documents.groupby('rd_norm').strict_split.nunique().max() == 1
strict_counts = documents[documents.strict_in_scope].strict_split.value_counts()
display(strict_counts.rename('documents').to_frame())

,documents
strict_split,
fit,712343
validation,281538
test,280989
calibration,140067


## 9. Baseline `tg_ids` и предел Recall

Для TG baseline говорит «да» всем её кандидатам. Если положительный Truth вообще не попал в candidates, двухэтапная модель не может его вернуть. Это и есть **candidate recall ceiling**.

In [10]:
def metric_row(y, pred, score=None):
    y = np.asarray(y, dtype=np.int8); pred = np.asarray(pred, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'pr_auc': float(average_precision_score(y, score)) if score is not None and len(np.unique(y)) > 1 else None,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'support': int(y.sum()), 'predicted_positive': int(pred.sum()), 'rows': int(len(y)),
    }

baseline_rows = []
test_part = documents.strict_split.eq('test')
for tg in TARGETS:
    universe = test_part & (documents[f'candidate_{tg}'].eq(1) | documents.truth_set.map(lambda s, g=tg: g in s))
    y = documents.loc[universe, 'truth_set'].map(lambda s, g=tg: int(g in s)).to_numpy()
    if not y.any():
        print(f'TG {tg}: candidate coverage/purity not evaluated: no linked truth positives.')
        continue
    pred = documents.loc[universe, f'candidate_{tg}'].to_numpy(dtype=bool)
    row = metric_row(y, pred, pred.astype(float)); row.update({'tg': tg, 'model': 'tg_ids baseline'})
    row['candidate_recall_ceiling'] = row['recall']
    baseline_rows.append(row)
baseline_strict = pd.DataFrame(baseline_rows).set_index('tg')
display(baseline_strict[['precision','recall','f1','support','predicted_positive']])

,precision,recall,f1,support,predicted_positive
tg,,,,,
4,0.018172,0.998728,0.035695,786,43198
35,0.051503,0.997946,0.097950,13631,264123
43,0.134526,0.984968,0.236721,2794,20457


## 10. Word/character TF-IDF, NB-SVM и структурная модель

Текстовая ветка:

- слова и биграммы названия;
- отдельные слова нормативного контекста;
- символьные 3–5-граммы field-aware текста — устойчивы к опечаткам, окончаниям и транслитерации;
- NB-SVM усиливает фрагменты, характерные для positives, и ослабляет характерные для negatives;
- `SGDClassifier` быстро обучает линейную модель на разреженной матрице.

Структурная ветка — `HistGradientBoostingClassifier` на компактных понятных признаках. Категории представлены частотами из train, без утечки из validation/test.

In [11]:
LOW_MEMORY_BATCH = 8_000

def fit_vectorizers(frame, max_rows=60_000):
    if len(frame) > max_rows:
        keep = frame.index.to_series().map(lambda x: stable_int(x) < max_rows / len(frame) * 10_000)
        sample = frame.loc[keep].head(max_rows)
    else:
        sample = frame
    vecs = {
        'product_word': TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=.998,
                                        max_features=30_000, sublinear_tf=True, dtype=np.float32),
        'context_word': TfidfVectorizer(ngram_range=(1,2), min_df=4, max_df=.998,
                                        max_features=15_000, sublinear_tf=True, dtype=np.float32),
        'field_char': TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=4,
                                      max_features=35_000, sublinear_tf=True, dtype=np.float32),
    }
    vecs['product_word'].fit(sample.product_text.fillna(''))
    vecs['context_word'].fit(sample.context_text.fillna(''))
    char_source = sample.product_text.fillna('').astype(str) + ' __ctx__ ' + sample.context_text.fillna('').astype(str)
    vecs['field_char'].fit(char_source)
    return vecs

def text_matrix(frame, vecs):
    return sparse.hstack([
        vecs['product_word'].transform(frame.product_text.fillna('')),
        vecs['context_word'].transform(frame.context_text.fillna('')),
        vecs['field_char'].transform(frame.product_text.fillna('').astype(str) + ' __ctx__ ' +
                                     frame.context_text.fillna('').astype(str)),
    ], format='csr', dtype=np.float32)

def fit_nbsvm(X, y):
    y = np.asarray(y, dtype=np.int8)
    pos = np.asarray(X[y == 1].mean(axis=0)).ravel() + 1e-4
    neg = np.asarray(X[y == 0].mean(axis=0)).ravel() + 1e-4
    ratio = np.log(pos / neg).astype(np.float32)
    npos, nneg = max(int(y.sum()), 1), max(int((1-y).sum()), 1)
    weights = np.where(y == 1, min(nneg / npos, 25.0), 1.0)
    model = SGDClassifier(loss='log_loss', penalty='elasticnet', l1_ratio=.05, alpha=2e-5,
                          max_iter=80, tol=1e-4, average=True, random_state=RANDOM_STATE)
    model.fit(X.multiply(ratio), y, sample_weight=weights)
    return {'model': model, 'ratio': ratio}

def score_nbsvm(bundle, X):
    return bundle['model'].decision_function(X.multiply(bundle['ratio']))

def fit_nbsvm_batches(frame, y, vecs, epochs=2):
    # Тот же NB-SVM, но ни один TF-IDF batch не остаётся в RAM целиком.
    y = np.asarray(y, dtype=np.int8)
    pos_sum = neg_sum = None; npos = nneg = 0
    for start in range(0, len(frame), LOW_MEMORY_BATCH):
        part = frame.iloc[start:start + LOW_MEMORY_BATCH]
        yb = y[start:start + LOW_MEMORY_BATCH]
        X = text_matrix(part, vecs)
        if pos_sum is None:
            pos_sum = np.zeros(X.shape[1], np.float64); neg_sum = np.zeros(X.shape[1], np.float64)
        if np.any(yb == 1): pos_sum += np.asarray(X[yb == 1].sum(axis=0)).ravel(); npos += int((yb == 1).sum())
        if np.any(yb == 0): neg_sum += np.asarray(X[yb == 0].sum(axis=0)).ravel(); nneg += int((yb == 0).sum())
        del X
    ratio = np.log(pos_sum / max(npos, 1) + 1e-4) - np.log(neg_sum / max(nneg, 1) + 1e-4)
    ratio = ratio.astype(np.float32)
    model = SGDClassifier(loss='log_loss', penalty='elasticnet', l1_ratio=.05, alpha=2e-5,
                          max_iter=1, tol=None, average=True, random_state=RANDOM_STATE)
    pos_weight = min(max(nneg, 1) / max(npos, 1), 25.0)
    for epoch in range(epochs):
        for start in range(0, len(frame), LOW_MEMORY_BATCH):
            part = frame.iloc[start:start + LOW_MEMORY_BATCH]
            yb = y[start:start + LOW_MEMORY_BATCH]
            X = text_matrix(part, vecs)
            weights = np.where(yb == 1, pos_weight, 1.0)
            model.partial_fit(X.multiply(ratio), yb, classes=np.array([0,1]), sample_weight=weights)
            del X
    gc.collect()
    return {'model': model, 'ratio': ratio}

def fit_frequency_maps(frame):
    return {col: frame[col].fillna('').astype(str).value_counts(normalize=True).to_dict() for col in CAT_COLS}

def structural_matrix(frame, maps):
    numeric = frame[STRUCT_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(np.float32)
    cats = np.column_stack([
        frame[col].fillna('').astype(str).map(maps[col]).fillna(0).to_numpy(np.float32)
        for col in CAT_COLS
    ])
    return np.hstack([numeric, cats]).astype(np.float32)

def fit_structural(X, y):
    y = np.asarray(y, dtype=np.int8)
    npos, nneg = max(int(y.sum()), 1), max(int((1-y).sum()), 1)
    weights = np.where(y == 1, min(nneg / npos, 25.0), 1.0)
    model = HistGradientBoostingClassifier(max_iter=110, learning_rate=.07, max_leaf_nodes=31,
                                           min_samples_leaf=30, l2_regularization=1.0,
                                           random_state=RANDOM_STATE)
    model.fit(X, y, sample_weight=weights)
    return model

def fit_calibrator(raw, y):
    model = LogisticRegression(C=1.0, random_state=RANDOM_STATE)
    model.fit(np.asarray(raw).reshape(-1,1), np.asarray(y, dtype=np.int8))
    return model

def calibrated(calibrator, raw):
    return calibrator.predict_proba(np.asarray(raw).reshape(-1,1))[:,1]

print('Функции моделей готовы.')

Функции моделей готовы.


## 11. Hard-negative mining

Случайные negatives часто слишком лёгкие. Два прохода делают обучение полезнее:

1. черновая модель видит все positives и случайные negatives;
2. она оценивает остальные train-кандидаты;
3. финальная модель получает самые опасные false positives плюс контрольную случайную выборку.

Validation и test при этом никогда не семплируются.

In [12]:
def deterministic_take(index, n, salt=''):
    index = pd.Index(index)
    if len(index) <= n: return index
    order = sorted(index, key=lambda x: stable_int(f'{salt}|{x}', 10**9))
    return pd.Index(order[:n])

def mine_training_indices(frame, tg, vecs, use_gate=True):
    y = frame.truth_set.map(lambda s, g=tg: int(g in s))
    gate = frame[f'candidate_{tg}'].eq(1) if use_gate else pd.Series(True, index=frame.index)
    universe = gate | y.eq(1)
    pos = frame.index[universe & y.eq(1)]
    neg = frame.index[universe & y.eq(0)]
    seed_neg = deterministic_take(neg, min(len(neg), max(8 * len(pos), 3000)), f'seed-{tg}')
    seed = pos.union(seed_neg)
    first = fit_nbsvm_batches(frame.loc[seed], y.loc[seed].to_numpy(), vecs, epochs=1)

    hard_parts = []
    batch = LOW_MEMORY_BATCH
    for start in range(0, len(neg), batch):
        ids = neg[start:start+batch]
        X_batch = text_matrix(frame.loc[ids], vecs)
        score = score_nbsvm(first, X_batch)
        del X_batch
        hard_parts.append(pd.Series(score, index=ids))
    hard_scores = pd.concat(hard_parts).sort_values(ascending=False) if hard_parts else pd.Series(dtype=float)
    hard = hard_scores.head(min(len(hard_scores), max(8 * len(pos), 5000), 60_000)).index
    control = deterministic_take(neg.difference(hard), min(len(neg), max(2 * len(pos), 2000), 20_000), f'control-{tg}')
    final_index = pos.union(hard).union(control)
    return final_index, first, {'positives': len(pos), 'seed_negatives': len(seed_neg),
                                'hard_negatives': len(hard), 'control_negatives': len(control)}

print('Hard-negative mining будет выполняться только внутри fit.')

Hard-negative mining будет выполняться только внутри fit.


## 12. Выбор ансамбля и порога

Калибровка переводит сырые scores двух разных моделей в сопоставимые вероятности. Затем validation перебирает веса ансамбля.

Для строгого теста сначала требуется, чтобы нижняя односторонняя 95%-граница Wilson для Recall была ≥97%. Если данных недостаточно, notebook честно переключается на наблюдаемый Recall ≥97% и помечает отсутствие статистического запаса.

In [13]:
def wilson_lower(tp, positives, z=1.6448536269514722):
    if positives == 0: return 0.0
    p = tp / positives
    den = 1 + z*z/positives
    return (p + z*z/(2*positives) - z*math.sqrt(p*(1-p)/positives + z*z/(4*positives**2))) / den

def best_threshold(y, score, gate, require_margin):
    y = np.asarray(y, dtype=np.int8); score = np.asarray(score, float); gate = np.asarray(gate, bool)
    order = np.where(gate)[0][np.argsort(-score[gate], kind='mergesort')]
    ys = y[order]; ss = score[order]
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys); positives = int(y.sum())
    distinct = np.r_[ss[1:] != ss[:-1], True]
    positions = np.where(distinct)[0]
    precision = tp[positions] / np.maximum(tp[positions] + fp[positions], 1)
    recall = tp[positions] / max(positives, 1)
    lower = np.array([wilson_lower(int(tp[i]), positives) for i in positions])
    eligible_margin = lower >= .97
    eligible_observed = recall >= .97
    eligible = eligible_margin if require_margin and eligible_margin.any() else eligible_observed
    rule = 'wilson_95_lower>=0.97' if require_margin and eligible_margin.any() else 'observed_recall>=0.97'
    if not eligible.any():
        eligible = recall == recall.max(); rule = 'target_unreachable_best_recall'
    candidates = positions[eligible]
    candidate_p = precision[eligible]
    best = candidates[np.argmax(candidate_p)]
    return {'threshold': float(ss[best]), 'validation_precision': float(tp[best] / (tp[best] + fp[best])),
            'validation_recall': float(tp[best] / max(positives,1)),
            'validation_recall_lower95': float(wilson_lower(int(tp[best]), positives)), 'rule': rule}

def choose_blend(y, p_text, p_struct, gate, require_margin):
    trials = []
    for weight in np.linspace(0, 1, 9):
        score = weight * p_text + (1-weight) * p_struct
        picked = best_threshold(y, score, gate, require_margin)
        picked['text_weight'] = float(weight)
        trials.append(picked)
    valid = [x for x in trials if x['validation_recall'] >= .97]
    pool = valid or trials
    best = max(pool, key=lambda x: (x['validation_precision'], x['validation_recall_lower95']))
    return best, trials

print('Сетка весов:', list(np.linspace(0, 1, 9)))

Сетка весов: [np.float64(0.0), np.float64(0.125), np.float64(0.25), np.float64(0.375), np.float64(0.5), np.float64(0.625), np.float64(0.75), np.float64(0.875), np.float64(1.0)]


## 13. Общая функция эксперимента

Она одинакова для legacy и strict: fit → hard negatives → calibration → validation → заморозка → test. Различаются только split и candidate gate.

In [14]:
def run_experiment(frame, split_col, use_gate, require_margin, label):
    fit = frame[frame[split_col].eq('fit')]
    cal = frame[frame[split_col].eq('calibration')]
    val = frame[frame[split_col].eq('validation')]
    test = frame[frame[split_col].eq('test')]
    assert set(fit.index).isdisjoint(val.index) and set(val.index).isdisjoint(test.index)
    print(label, {x: len(z) for x,z in [('fit',fit),('calibration',cal),('validation',val),('test',test)]})

    vec_fit = fit
    if len(fit) > 60_000:
        any_positive = fit.truth_set.map(lambda s: bool(s & set(TARGETS)))
        bg = fit.index[~any_positive]
        chosen_bg = deterministic_take(bg, 45_000, label+'-vec')
        vec_fit = fit.loc[fit.index[any_positive].union(chosen_bg)]
    vecs = fit_vectorizers(vec_fit)
    freq_maps = fit_frequency_maps(fit)
    models, calibrators, frozen, validation_trials, mining = {}, {}, {}, {}, {}
    test_rows, baseline_rows_local = [], []
    test_scores = pd.DataFrame(index=test.index)

    target_status = {}
    for tg in TARGETS:
        split_positive_counts = {
            name: int(part.truth_set.map(lambda s, g=tg: g in s).sum())
            for name, part in [('fit', fit), ('calibration', cal), ('validation', val), ('test', test)]
        }
        # No linked positives means this is not a supervised task. Never fit a one-class model.
        if not all(split_positive_counts.values()):
            target_status[str(tg)] = {
                'status': 'not_supervised_unlinked_or_insufficient_labels',
                'positive_counts': split_positive_counts,
            }
            print(f'  TG {tg}: supervised evaluation skipped; positives by split={split_positive_counts}')
            continue
        target_status[str(tg)] = {'status': 'supervised', 'positive_counts': split_positive_counts}
        print(f'  TG {tg}: mining and fit')
        train_ids, first, mine_info = mine_training_indices(fit, tg, vecs, use_gate)
        mining[str(tg)] = mine_info
        y_train = fit.loc[train_ids, 'truth_set'].map(lambda s, g=tg: int(g in s)).to_numpy()
        text_model = fit_nbsvm_batches(fit.loc[train_ids], y_train, vecs, epochs=2)
        X_struct_train = structural_matrix(fit.loc[train_ids], freq_maps)
        struct_model = fit_structural(X_struct_train, y_train)
        models[str(tg)] = {'text': text_model, 'structural': struct_model}

        def scores(part):
            # Важно для обычного ноутбука: никогда не строим char-TF-IDF всей части сразу.
            text_parts, struct_parts = [], []
            for start in range(0, len(part), LOW_MEMORY_BATCH):
                batch_part = part.iloc[start:start + LOW_MEMORY_BATCH]
                X_text = text_matrix(batch_part, vecs)
                text_parts.append(score_nbsvm(text_model, X_text))
                del X_text
                X_struct = structural_matrix(batch_part, freq_maps)
                struct_parts.append(struct_model.predict_proba(X_struct)[:,1])
                del X_struct
            gc.collect()
            return np.concatenate(text_parts), np.concatenate(struct_parts)

        cal_y = cal.truth_set.map(lambda s, g=tg: int(g in s)).to_numpy()
        cal_gate = cal[f'candidate_{tg}'].eq(1).to_numpy() if use_gate else np.ones(len(cal), bool)
        cal_universe = cal_gate | cal_y.astype(bool)
        raw_tc, raw_sc = scores(cal.loc[cal_universe])
        gate_inside = cal_gate[cal_universe]
        # Калибратор учится только на документах, которые реально пропускает candidate stage.
        y_cal_u = cal_y[cal_universe]
        text_cal = fit_calibrator(raw_tc[gate_inside], y_cal_u[gate_inside])
        struct_cal = fit_calibrator(raw_sc[gate_inside], y_cal_u[gate_inside])
        calibrators[str(tg)] = {'text': text_cal, 'structural': struct_cal}

        val_y_all = val.truth_set.map(lambda s, g=tg: int(g in s)).to_numpy()
        val_gate_all = val[f'candidate_{tg}'].eq(1).to_numpy() if use_gate else np.ones(len(val), bool)
        val_universe = val_gate_all | val_y_all.astype(bool)
        val_u = val.loc[val_universe]
        val_y = val_y_all[val_universe]; val_gate = val_gate_all[val_universe]
        raw_tv, raw_sv = scores(val_u)
        pt = calibrated(text_cal, raw_tv); ps = calibrated(struct_cal, raw_sv)
        choice, trials = choose_blend(val_y, pt, ps, val_gate, require_margin)
        frozen[str(tg)] = choice; validation_trials[str(tg)] = trials

        test_y_all = test.truth_set.map(lambda s, g=tg: int(g in s)).to_numpy()
        test_gate_all = test[f'candidate_{tg}'].eq(1).to_numpy() if use_gate else np.ones(len(test), bool)
        test_universe = test_gate_all | test_y_all.astype(bool)
        test_u = test.loc[test_universe]
        y = test_y_all[test_universe]; gate = test_gate_all[test_universe]
        raw_tt, raw_st = scores(test_u)
        pt = calibrated(text_cal, raw_tt); ps = calibrated(struct_cal, raw_st)
        final_score = choice['text_weight'] * pt + (1-choice['text_weight']) * ps
        pred = gate & (final_score >= choice['threshold'])
        row = metric_row(y, pred, final_score)
        row.update({'tg': tg, 'model': 'v4 ensemble', 'threshold': choice['threshold'],
                    'text_weight': choice['text_weight'], 'threshold_rule': choice['rule']})
        test_rows.append(row)

        base_pred = test_u[f'candidate_{tg}'].eq(1).to_numpy() if use_gate else test_u[f'candidate_{tg}'].eq(1).to_numpy()
        base = metric_row(y, base_pred, base_pred.astype(float)); base.update({'tg':tg,'model':'tg_ids baseline'})
        baseline_rows_local.append(base)
        test_scores.loc[test_u.index, f'{label}_score_{tg}'] = final_score
        test_scores.loc[test_u.index, f'{label}_pred_{tg}'] = pred.astype('int8')
        test_scores.loc[test_u.index, f'{label}_truth_{tg}'] = y

    return {
        'vectorizers': vecs, 'frequency_maps': freq_maps, 'models': models,
        'calibrators': calibrators, 'frozen': frozen, 'validation_trials': validation_trials,
        'mining': mining, 'target_status': target_status, 'test_metrics': pd.DataFrame(test_rows),
        'baseline_metrics': pd.DataFrame(baseline_rows_local), 'test_scores': test_scores,
    }

## 14. Legacy benchmark — честное сравнение с v1

Повторяем v1 буквально:

- все Truth;
- 5% target-кандидатов;
- 0,1% background;
- caps `other=20k`, `35=20k`, `43=15k`, `4=10k`;
- stratified random split 60/20/20, random state 42.

Multi-label не выбрасываем из общей витрины, но исключаем именно из legacy, потому что v1 обучал single-label задачу. v5-модель для этого benchmark обучается отдельно и не видит legacy-test.

In [15]:
legacy_label = documents.truth_set.map(
    lambda s: str(next(iter(s))) if len(s) == 1 else ('MULTI' if len(s) > 1 else 'other'))
hard = documents.candidate_set.map(lambda s: bool(s & set(TARGETS)))
bucket = documents.rd_norm.map(stable_int)
legacy_mask = (documents.truth_set.map(len).gt(0)) | (hard & bucket.lt(500)) | (~hard & bucket.lt(10))
legacy = documents.loc[legacy_mask].copy()
legacy['legacy_label'] = legacy_label.loc[legacy.index]
legacy = legacy[legacy.legacy_label.ne('MULTI')]
caps = {'other':20_000, '35':20_000, '43':15_000, '4':10_000}
legacy = pd.concat([
    g.sample(min(len(g), caps.get(label, 10_000)), random_state=RANDOM_STATE)
    for label, g in legacy.groupby('legacy_label')
], ignore_index=False).sample(frac=1, random_state=RANDOM_STATE)

legacy_train_val, legacy_test = train_test_split(
    legacy, test_size=.20, stratify=legacy.legacy_label, random_state=RANDOM_STATE)
legacy_train, legacy_val = train_test_split(
    legacy_train_val, test_size=.25, stratify=legacy_train_val.legacy_label, random_state=RANDOM_STATE)
legacy_fit, legacy_cal = train_test_split(
    legacy_train, test_size=.20, stratify=legacy_train.legacy_label, random_state=RANDOM_STATE)
legacy_fit = pd.concat([legacy_fit, legacy_cal.iloc[0:0]])
legacy.loc[legacy_fit.index, 'legacy_split'] = 'fit'
legacy.loc[legacy_cal.index, 'legacy_split'] = 'calibration'
legacy.loc[legacy_val.index, 'legacy_split'] = 'validation'
legacy.loc[legacy_test.index, 'legacy_split'] = 'test'
assert legacy.legacy_split.notna().all()

legacy_result = run_experiment(legacy, 'legacy_split', use_gate=False,
                               require_margin=False, label='legacy')
legacy_metrics = legacy_result['test_metrics'].set_index('tg')
legacy_baseline = legacy_result['baseline_metrics'].set_index('tg')
for tg in legacy_metrics.index:
    legacy_metrics.loc[tg, 'v1_precision'] = V1[tg]['precision']
    legacy_metrics.loc[tg, 'v1_recall'] = V1[tg]['recall']
    legacy_metrics.loc[tg, 'v1_f1'] = V1[tg]['f1']
    legacy_metrics.loc[tg, 'passes_v1_acceptance'] = bool(
        legacy_metrics.loc[tg,'recall'] >= .97 and
        legacy_metrics.loc[tg,'precision'] > V1[tg]['precision'] and
        legacy_metrics.loc[tg,'f1'] > V1[tg]['f1'])
display(legacy_metrics[['precision','recall','f1','v1_precision','v1_recall','v1_f1','passes_v1_acceptance']])

legacy {'fit': 27806, 'calibration': 6952, 'validation': 11586, 'test': 11586}
  TG 4: mining and fit
  TG 35: mining and fit
  TG 43: mining and fit


,precision,recall,f1,v1_precision,v1_recall,v1_f1,passes_v1_acceptance
tg,,,,,,,
4,0.945776,0.960307,0.952986,0.153263,0.983355,0.265193,False
35,0.857488,0.974750,0.912367,0.509449,0.977250,0.669751,True
43,0.977849,0.975758,0.976802,0.906489,0.971123,0.937694,True


## 15. Strict full-candidate evaluation

Теперь основной строгий контур оценки. Для каждой TG validation и test содержат **всех** её кандидатов и все positives. Никаких caps и sampling. Candidate gate явно ограничивает достижимый Recall.

In [16]:
MODEL_KEEP = sorted(set(['rd_number','candidate_tgs','truth_tgs','product_text','context_text',
                         'truth_set','candidate_set','strict_split'] + STRUCT_COLS + CAT_COLS +
                        [f'candidate_{tg}' for tg in TARGETS]))
strict = documents[MODEL_KEEP].copy()
N_DOCUMENTS = len(documents)
del documents, legacy_label, hard, bucket
gc.collect()
strict_result = run_experiment(strict, 'strict_split', use_gate=True,
                               require_margin=True, label='strict')
supervised_targets = tuple(int(tg) for tg, status in strict_result['target_status'].items()
                           if status['status'] == 'supervised')
strict_metrics = strict_result['test_metrics'].set_index('tg')
strict_baseline = strict_result['baseline_metrics'].set_index('tg')
for tg in strict_metrics.index:
    strict_metrics.loc[tg, 'baseline_precision'] = strict_baseline.loc[tg, 'precision']
    strict_metrics.loc[tg, 'baseline_recall'] = strict_baseline.loc[tg, 'recall']
    strict_metrics.loc[tg, 'candidate_recall_ceiling'] = strict_baseline.loc[tg, 'recall']
    strict_metrics.loc[tg, 'beats_same_test_baseline'] = bool(
        strict_metrics.loc[tg,'recall'] >= .97 and
        strict_metrics.loc[tg,'precision'] > strict_baseline.loc[tg,'precision'])
display(strict_metrics[['precision','recall','f1','pr_auc','baseline_precision',
                        'candidate_recall_ceiling','threshold_rule','beats_same_test_baseline']])

strict {'fit': 713404, 'calibration': 140274, 'validation': 281960, 'test': 281391}
  TG 4: mining and fit
  TG 35: mining and fit
  TG 43: mining and fit


,precision,recall,f1,pr_auc,baseline_precision,candidate_recall_ceiling,threshold_rule,beats_same_test_baseline
tg,,,,,,,,
4,0.183996,0.977099,0.309677,0.573071,0.018172,0.998728,wilson_95_lower>=0.97,True
35,0.226200,0.970582,0.366894,0.602237,0.051503,0.997946,wilson_95_lower>=0.97,True
43,0.350522,0.973157,0.515401,0.730535,0.134526,0.984968,wilson_95_lower>=0.97,True


## 16. Проверки заморозки и разбор ошибок

Здесь assertions защищают от самых опасных методологических ошибок. Затем выводим false positives/false negatives и семантически подозрительные positive-метки. Они идут в audit, но Truth вручную не исправляется.

In [17]:
# Validation/test строгого контура — полные части, не результат sample/head.
for tg in supervised_targets:
    expected_val = strict[(strict.strict_split.eq('validation')) &
                          (strict[f'candidate_{tg}'].eq(1) | strict.truth_set.map(lambda s, g=tg: g in s))]
    expected_test = strict[(strict.strict_split.eq('test')) &
                           (strict[f'candidate_{tg}'].eq(1) | strict.truth_set.map(lambda s, g=tg: g in s))]
    assert strict_result['test_metrics'].set_index('tg').loc[tg, 'rows'] == len(expected_test)
    assert strict_result['frozen'][str(tg)]['threshold'] == strict_metrics.loc[tg, 'threshold']

score_frame = strict_result['test_scores']
error_examples, suspected_labels = {}, []
for tg in supervised_targets:
    part = strict.loc[score_frame[f'strict_truth_{tg}'].dropna().index].copy()
    part['score'] = score_frame.loc[part.index, f'strict_score_{tg}']
    part['pred'] = score_frame.loc[part.index, f'strict_pred_{tg}'].astype(int)
    part['truth'] = score_frame.loc[part.index, f'strict_truth_{tg}'].astype(int)
    fp = part[(part.pred.eq(1)) & (part.truth.eq(0))].nlargest(20, 'score')
    fn = part[(part.pred.eq(0)) & (part.truth.eq(1))].nsmallest(20, 'score')
    error_examples[str(tg)] = {
        'false_positive': fp[['rd_number','product_text','candidate_tgs','truth_tgs','score']].to_dict('records'),
        'false_negative': fn[['rd_number','product_text','candidate_tgs','truth_tgs','score']].to_dict('records'),
    }
    # Положительная метка подозрительна, если два чужих словаря сильнее целевого.
    positives = part[part.truth.eq(1)].copy()
    other = [f'term_score_{g}' for g in TARGETS if g != tg]
    suspicious = positives[positives[other].max(axis=1) > positives[f'term_score_{tg}'] + 2].head(15)
    for r in suspicious.itertuples():
        suspected_labels.append({'tg': tg, 'rd_number': r.rd_number,
                                 'product_text': r.product_text[:500], 'reason': 'чужой словарь заметно сильнее'})

for tg in supervised_targets:
    print('\nTG', tg, 'false positives / false negatives')
    display(pd.DataFrame(error_examples[str(tg)]['false_positive']).head(3))
    display(pd.DataFrame(error_examples[str(tg)]['false_negative']).head(3))

del strict
gc.collect()


TG 4 false positives / false negatives


,rd_number,product_text,candidate_tgs,truth_tgs,score
0,ЕАЭС N RU Д-RU.РА10.В.50500/24,средства парфюмерно-косметические жидкие: духи...,"4,35",,0.627336
1,ЕАЭС N RU Д-RU.РА03.В.78769/23,"парфюмерная вода, духи, распив. торговая марка...","4,35",,0.619654
2,ЕАЭС N RU Д-RU.РА02.В.09347/24,средства парфюмерно-косметические: масляные ду...,"4,35",,0.595252


,rd_number,product_text,candidate_tgs,truth_tgs,score
0,ЕАЭС N RU Д-FR.РА01.В.84379/21,продукция косметическая для ухода за кожей: “m...,"4,35",4,0.001297
1,ЕАЭС N RU Д-RU.ТР06.В.04364,продукция парфюмерная жидкая: туалетные воды д...,"4,35,37",4,0.001465
2,ЕАЭС N RU Д-RU.РА01.В.15575/21,"продукция косметическая гигиеническая моющая, ...","4,35","4,35",0.001695



TG 35 false positives / false negatives


,rd_number,product_text,candidate_tgs,truth_tgs,score
0,RU.77.99.88.002.Е.012908.05.11,"средство дезинфицирующее ""химитек антисептик-с...","4,35",,0.773059
1,ЕАЭС N RU Д-RU.РА06.В.11259/22,"продукция косметическая, в том числе в наборах...","4,35",,0.768588
2,ЕАЭС N RU Д-FR.РА05.В.81438/23,продукция косметическая для ухода за кожей мар...,"4,35",,0.761807


,rd_number,product_text,candidate_tgs,truth_tgs,score
0,ЕАЭС N RU Д-RU.РА01.В.98261/25,изделия трикотажные верхние второго слоя для в...,"1,35",35,0.001792
1,ЕАЭС RU С-CN.АЯ46.В.41126/25,одежда верхняя (2-го слоя) для детей старше 3-...,"1,35",35,0.001806
2,ЕАЭС N RU Д-CN.РА01.В.02933/24,изделия верхние трикотажные второго слоя для в...,"1,35,37",35,0.001835



TG 43 false positives / false negatives


,rd_number,product_text,candidate_tgs,truth_tgs,score
0,ЕАЭС N RU Д-CN.РА01.В.61449/24,масла моторные универсальные синтетические и п...,"35,43",,0.721664
1,ЕАЭС N RU Д-RU.РА07.В.50538/25,"масла моторные, торговая марка: «sinica». упак...",43,,0.714738
2,ЕАЭС N RU Д-AE.РА05.В.27809/25,"моторное масло марки согласно приложению no1, ...",43,,0.708691


,rd_number,product_text,candidate_tgs,truth_tgs,score
0,ЕАЭС N RU Д-RU.РА01.В.36650/21,охлаждающие жидкости kamaz g-profi service lin...,"8,35,37,43",43,0.006578
1,ЕАЭС N RU Д-DE.МБ32.В.15548,"масло трансмиссионное минеральное, условия хра...",43,43,0.009433
2,ЕАЭС N RU Д-KR.МБ32.В.14517,"смазка для резиновых уплотнений, срок службы у...",43,43,0.009522


172

## 17. Сохранение ровно трёх артефактов

В parquet добавляются split, финальные strict scores и error-флаги. Joblib содержит всё, что нужно для повторного применения. JSON хранит только проверяемый отчёт и небольшие примеры ошибок.

In [18]:
final_arrays = {}
for tg in supervised_targets:
    idx = strict_result['test_scores'][f'strict_score_{tg}'].dropna().index.to_numpy()
    scores = np.full(N_DOCUMENTS, np.nan, dtype=np.float32)
    fp = np.zeros(N_DOCUMENTS, dtype=np.int8); fn = np.zeros(N_DOCUMENTS, dtype=np.int8)
    scores[idx] = strict_result['test_scores'].loc[idx, f'strict_score_{tg}'].to_numpy(np.float32)
    pred = strict_result['test_scores'].loc[idx, f'strict_pred_{tg}'].to_numpy(bool)
    truth_y = strict_result['test_scores'].loc[idx, f'strict_truth_{tg}'].to_numpy(bool)
    fp[idx[pred & ~truth_y]] = 1; fn[idx[~pred & truth_y]] = 1
    final_arrays[f'final_score_{tg}'] = scores
    final_arrays[f'error_fp_{tg}'] = fp; final_arrays[f'error_fn_{tg}'] = fn

# Сначала сохраняем модели и отчёт. Даже если файловая система задержит parquet,
# дорогой ML-расчёт уже не потеряется.
bundle = {
    'version': CACHE_VERSION, 'targets': TARGETS, 'target_names': TARGET_NAMES,
    'definition_terms': definition_terms, 'definition_codes': definition_codes,
    'group_prefixes_candidate_rules': GROUP_PREFIXES, 'truth_linkage_audit': truth_linkage_audit.to_dict('index'),
    'struct_columns': STRUCT_COLS, 'categorical_columns': CAT_COLS,
    'legacy': {k:v for k,v in legacy_result.items() if k not in {'test_metrics','baseline_metrics','test_scores'}},
    'strict': {k:v for k,v in strict_result.items() if k not in {'test_metrics','baseline_metrics','test_scores'}},
}
joblib.dump(bundle, BUNDLE_PATH, compress=3)

def records(df):
    return json.loads(df.reset_index().to_json(orient='records', force_ascii=False))

report = {
    'versions': versions, 'random_state': RANDOM_STATE, 'audit': audit, 'truth_linkage_audit': records(truth_linkage_audit),
    'tg4_embedded_id_links_diagnostic_only': len(tg4_embedded_id_links),
    'group_prefixes_candidate_rules': {str(k): list(v) for k, v in GROUP_PREFIXES.items()},
    'legacy_v1_reference': V1,
    'legacy_metrics': records(legacy_metrics),
    'legacy_baseline': records(legacy_baseline),
    'strict_candidate_metrics': records(strict_metrics),
    'strict_candidate_baseline': records(strict_baseline),
    'validation_experiments': {
        'legacy': legacy_result['validation_trials'], 'strict': strict_result['validation_trials']},
    'hard_negative_mining': {'legacy': legacy_result['mining'], 'strict': strict_result['mining']},
    'frozen_thresholds': {'legacy': legacy_result['frozen'], 'strict': strict_result['frozen']},
    'errors': error_examples, 'suspected_label_issues': suspected_labels,
    'all_legacy_targets_win': bool(legacy_metrics.passes_v1_acceptance.all()),
    'all_strict_targets_pass': bool(strict_metrics.beats_same_test_baseline.all()),
}
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

# Потоково переписываем parquet: в RAM находится один row group, а не вся витрина.
temp_final = ARTIFACTS / '_prepared_documents.writing.parquet'
pf_final = pq.ParquetFile(PREPARED_PATH)
writer = None; offset = 0
for rg in range(pf_final.num_row_groups):
    table = pf_final.read_row_group(rg)
    n = len(table)
    # При повторном запуске заменяем старые score-колонки, а не дублируем их.
    keep = [name for name in table.schema.names if name not in final_arrays]
    table = table.select(keep)
    for name, values in final_arrays.items():
        table = table.append_column(name, pa.array(values[offset:offset+n]))
    if writer is None:
        writer = pq.ParquetWriter(temp_final, table.schema, compression='snappy')
    writer.write_table(table); offset += n
writer.close()
pf_final.close()
del pf_final, table
gc.collect()
os.replace(temp_final, PREPARED_PATH)

assert {p.name for p in ARTIFACTS.iterdir()} == {
    'prepared_documents.parquet', 'solution_bundle.joblib', 'report.json'}
print('Сохранено:')
for p in (PREPARED_PATH, BUNDLE_PATH, REPORT_PATH):
    print(' ', p.name, round(p.stat().st_size / 2**20, 1), 'MiB')

Сохранено:
  prepared_documents.parquet 1483.3 MiB
  solution_bundle.joblib 8.2 MiB
  report.json 0.3 MiB


## 18. Краткий аудит TNVED 3808

`3808` добавлен для TG35 как структурный feature через `tnved_prefix_35`. Он не формирует ground truth и не является самостоятельным candidate gate.

Ниже проверяется его фактическое покрытие и наблюдаемая специфичность только на документах с linked Truth.

In [19]:
audit_3808 = pd.read_parquet(
    PREPARED_PATH,
    columns=['truth_tgs', 'tnved_codes'],
    dtype_backend='pyarrow',
)

audit_3808['is_tg35_truth'] = (
    audit_3808['truth_tgs']
    .fillna('')
    .astype('string')
    .str.split(',')
    .apply(lambda xs: '35' in xs)
)

audit_3808['has_3808'] = (
    audit_3808['tnved_codes']
    .fillna('')
    .astype('string')
    .str.split(',')
    .apply(
        lambda codes: any(
            code.startswith('3808')
            for code in codes
            if code
        )
    )
)

linked = audit_3808[
    audit_3808['truth_tgs'].fillna('').astype('string').str.strip().ne('')
].copy()

linked_3808 = linked[linked['has_3808']]

print('Linked TG35 docs:', int(audit_3808['is_tg35_truth'].sum()))
print(
    'TG35 docs with 3808-family:',
    int((audit_3808['is_tg35_truth'] & audit_3808['has_3808']).sum())
)
print(
    'Coverage among linked TG35:',
    round(
        audit_3808.loc[audit_3808['is_tg35_truth'], 'has_3808'].mean(),
        4
    )
)
print(
    'Linked docs with 3808-family:',
    len(linked_3808)
)
print(
    'Observed purity for pure TG35 among linked 3808 docs:',
    round((linked_3808['truth_tgs'] == '35').mean(), 4)
)


Linked TG35 docs: 68370
TG35 docs with 3808-family: 1055
Coverage among linked TG35: 0.0154
Linked docs with 3808-family: 1056
Observed purity for pure TG35 among linked 3808 docs: 0.9981


In [20]:
legacy_ok = bool(legacy_metrics.passes_v1_acceptance.all()) if not legacy_metrics.empty else False
strict_ok = bool(strict_metrics.beats_same_test_baseline.all()) if not strict_metrics.empty else False

print(
    'TG4 linkage status:',
    truth_linkage_audit.loc[4, 'supervised_status']
)
print('Legacy acceptance criterion passed for all supervised TG:', legacy_ok)
print('Strict model improves Precision over tg_ids baseline for all supervised TG:', strict_ok)

if legacy_ok and strict_ok:
    print('\nИТОГ v5: оба критерия выполнены.')
else:
    print('\nИТОГ v5: оба критерия одновременно не выполнены — это фиксируется без подгонки результата.')
    for tg in legacy_metrics.index:
        if not legacy_metrics.loc[tg, 'passes_v1_acceptance']:
            print(f'  TG {tg}: legacy не прошёл acceptance criterion относительно v1.')
    for tg in strict_metrics.index:
        if not strict_metrics.loc[tg, 'beats_same_test_baseline']:
            print(
                f"  TG {tg}: strict не улучшил Precision относительно tg_ids baseline; "
                f"candidate ceiling={strict_metrics.loc[tg,'candidate_recall_ceiling']:.3%}."
            )

display(
    legacy_metrics[['precision','recall','f1','passes_v1_acceptance']]
    .style.format('{:.3f}', subset=['precision','recall','f1'])
)

display(
    strict_metrics[['precision','recall','f1','candidate_recall_ceiling','beats_same_test_baseline']]
    .style.format('{:.3f}', subset=['precision','recall','f1','candidate_recall_ceiling'])
)


TG4 linkage status: linked
Legacy acceptance criterion passed for all supervised TG: False
Strict model improves Precision over tg_ids baseline for all supervised TG: True

ИТОГ v5: оба критерия одновременно не выполнены — это фиксируется без подгонки результата.
  TG 4: legacy не прошёл acceptance criterion относительно v1.


,precision,recall,f1,passes_v1_acceptance
tg,,,,
4,0.946,0.960,0.953,False
35,0.857,0.975,0.912,True
43,0.978,0.976,0.977,True


,precision,recall,f1,candidate_recall_ceiling,beats_same_test_baseline
tg,,,,,
4,0.184,0.977,0.310,0.999,True
35,0.226,0.971,0.367,0.998,True
43,0.351,0.973,0.515,0.985,True


### Ограничения v5

- Strict test оценивает выбранный после exploratory feature discovery набор признаков; отдельная out-of-sample feature-selection процедура не проводилась.
- `3808` имеет высокую наблюдаемую специфичность на linked Truth, но его полноценный standalone target-vs-OTHER audit и retraining ablation `with/without 3808` не выполнялись.
- Для финальных выводов используются только фактически достигнутые критерии, без подгонки под V1.